# ECA-LDNet — Stage 4: Transfer Learning Fine-Tuning
## Perceptual Refinement from Stage 3 Best Model

**Objective:** Load `stage3_best.keras` and continue training with advanced perceptual losses to maximize PSNR, SSIM, and MS-SSIM while preserving the lightweight architecture (1.482M params).

### Stage 4 Strategy
- **Transfer Learning:** All weights initialized from Stage 3 best checkpoint
- **Optimizer:** AdamW with Cosine Decay (LR: 1e-5 → 1e-7)
- **EMA:** Exponential Moving Average (decay=0.999) for stable validation
- **Loss:** Multi-component perceptual refinement loss with 8 terms
- **Max Epochs:** 30 with early stopping (patience=8)



In [19]:
# ═══════════════════════════════════════════════════════════════
# SECTION 1 — IMPORTS & GPU SETUP
# ═══════════════════════════════════════════════════════════════
import os, sys, gc, csv, time, random, warnings, json
import numpy as np
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("float32")
tf.keras.backend.set_floatx("float32")
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import VGG19

# ─── Mixed Precision ───
# Stage 4 uses float32 only — mixed_float16 removed intentionally
# (VGG perceptual loss + MS-SSIM are numerically sensitive at LR 1e-5→1e-7)
print(f"Mixed Precision Policy: {tf.keras.mixed_precision.global_policy().name}")

# ─── GPU Setup ───
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("=" * 70)
print("ECA-LDNet — Stage 4: Transfer Learning Fine-Tuning")
print("=" * 70)
print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {len(gpus)}")
if gpus:
    print(f"GPU name   : {gpus[0].name}")

# ─── Seeds ───
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─── Config ───
IMG_SIZE    = 256
CHANNELS    = 3
BATCH_SIZE  = 16
MAX_EPOCHS  = 30
INIT_LR     = 1e-5
MIN_LR      = 1e-7
WEIGHT_DECAY= 1e-5
EMA_DECAY   = 0.999
PATIENCE    = 8
CLIP_NORM   = 1.0

print(f"\nConfig: IMG={IMG_SIZE}, BS={BATCH_SIZE}, Epochs={MAX_EPOCHS}")
print(f"LR: {INIT_LR} → {MIN_LR}, WD={WEIGHT_DECAY}, EMA={EMA_DECAY}")



Mixed Precision Policy: float32
ECA-LDNet — Stage 4: Transfer Learning Fine-Tuning
TensorFlow : 2.19.0
GPUs found : 1
GPU name   : /physical_device:GPU:0

Config: IMG=256, BS=16, Epochs=30
LR: 1e-05 → 1e-07, WD=1e-05, EMA=0.999


In [20]:
import keras
import tensorflow as tf

print("Keras:", keras.__version__)
print("TensorFlow:", tf.__version__)

Keras: 3.13.2
TensorFlow: 2.19.0


In [21]:
import tensorflow as tf
from tensorflow.keras import layers

# ═══════════════════════════════════════════════════════════════
# CUSTOM LAYERS — KERAS 3 SAFE VERSION (NO ARCHITECTURE CHANGE)
# ═══════════════════════════════════════════════════════════════
class ECABlock(layers.Layer):
    """Efficient Channel Attention — learns WHICH channels matter."""
    def __init__(self, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.conv = layers.Conv1D(1, self.kernel_size, padding='same', use_bias=False)
        self.gap = layers.GlobalAveragePooling2D()
    def call(self, x):
        x = tf.cast(x, tf.float32)                              # ← ADDED
        B = tf.shape(x)[0]
        gap = self.gap(x)
        gap = tf.reshape(gap, [B, self.channels, 1])
        attn = tf.sigmoid(tf.cast(self.conv(gap), tf.float32))  # ← ADDED cast
        attn = tf.reshape(attn, [B, 1, 1, self.channels])
        return x * attn
    def get_config(self):
        cfg = super().get_config()
        cfg['kernel_size'] = self.kernel_size
        return cfg

class PixelAttention(layers.Layer):
    """Spatial Attention — learns WHERE to focus."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    def build(self, input_shape):
        C = input_shape[-1]
        self.conv1 = layers.Conv2D(max(8, C // 4), 1, padding='same', use_bias=False)
        self.conv2 = layers.Conv2D(1, 1, padding='same', activation='sigmoid')
    def call(self, x):
        x = tf.cast(x, tf.float32)                              # ← ADDED
        attn = tf.nn.relu(tf.cast(self.conv1(x), tf.float32))  # ← ADDED cast
        attn = tf.cast(self.conv2(attn), tf.float32)            # ← ADDED cast
        return x * attn
    def get_config(self):
        return super().get_config()

class PhysicsCorrectionLayer(layers.Layer):
    """Atmospheric scattering model correction: J(x) = (I(x) - A) / t(x) + A"""
    def __init__(self, eps=0.1, blend=0.08, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps
        self.blend = blend
    def call(self, inputs):
        img, A, t = inputs
        img = tf.cast(img, tf.float32)   # ← ADDED
        A   = tf.cast(A,   tf.float32)   # ← ADDED
        t   = tf.cast(t,   tf.float32)   # ← ADDED
        A_bc = tf.reshape(A, [-1, 1, 1, 3])
        A_bc = tf.broadcast_to(A_bc, tf.shape(img))
        t_bc = tf.broadcast_to(t, tf.shape(img))
        j = (img - A_bc) / (t_bc + self.eps) + A_bc
        j = tf.clip_by_value(j, 0.0, 1.0)
        return img * (1 - self.blend) + j * self.blend
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'eps': self.eps, 'blend': self.blend})
        return cfg

CUSTOM_OBJECTS = {
    'ECABlock': ECABlock,
    'PixelAttention': PixelAttention,
    'PhysicsCorrectionLayer': PhysicsCorrectionLayer,
}


In [22]:
# ═══════════════════════════════════════════════════════════════
# SECTION 3 — LOAD STAGE 3 BEST MODEL
# ═══════════════════════════════════════════════════════════════
import glob

# Auto-detect model path
MODEL_CANDIDATES = [
    '/kaggle/input/models/codeninjalucky/stage-4-model/tensorflow2/default/1/stage4_best.keras',
    # '/kaggle/working/stage3_best.keras',
    # '/kaggle/working/models/stage3_best.keras',
    # '/kaggle/input/eca-ldnet-model-train/stage3_best.keras',
]

# Also search any input datasets
for root_dir in glob.glob('/kaggle/input/*'):
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for f in filenames:
            if f == 'stage4_best.keras':
                MODEL_CANDIDATES.append(os.path.join(dirpath, f))

MODEL_PATH = None
for c in MODEL_CANDIDATES:
    if os.path.exists(c):
        MODEL_PATH = c
        break

if MODEL_PATH is None:
    raise FileNotFoundError(
        "stage4_best.keras not found! Searched:\n" +
        "\n".join(MODEL_CANDIDATES) +
        "\nPlease upload stage4_best.keras to your Kaggle dataset."
    )

print(f"Loading Stage 4 model: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH, custom_objects=CUSTOM_OBJECTS, compile=False)

total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"✓ Model loaded successfully!")
print(f"  Total params    : {total_params:,} ({total_params/1e6:.3f}M)")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Layers          : {len(model.layers)}")
print(f"  Input shape     : {model.input_shape}")
print(f"  Output shape    : {model.output_shape}")

# Verify ALL layers are trainable (Stage 4: freeze NONE)
for layer in model.layers:
    layer.trainable = True
print(f"\n✓ All {len(model.layers)} layers set to trainable")



Loading Stage 4 model: /kaggle/input/models/codeninjalucky/stage-4-model/tensorflow2/default/1/stage4_best.keras
✓ Model loaded successfully!
  Total params    : 1,481,871 (1.482M)
  Trainable params: 1,466,985
  Layers          : 206
  Input shape     : (None, 256, 256, 3)
  Output shape    : (None, 256, 256, 3)

✓ All 206 layers set to trainable


In [23]:
# ═══════════════════════════════════════════════════════════════
# SECTION 4 — DATASET PATHS & DATA PIPELINE
# ═══════════════════════════════════════════════════════════════
import cv2

def find_dir(base, candidates):
    """Auto-detect dataset directories."""
    if not base or not os.path.isdir(base):
        return None
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p) and len(os.listdir(p)) > 0:
            return p
    for sub in os.listdir(base):
        sp = os.path.join(base, sub)
        if os.path.isdir(sp):
            for c in candidates:
                p = os.path.join(sp, c)
                if os.path.isdir(p) and len(os.listdir(p)) > 0:
                    return p
    return None

# ─── RESIDE-6K ───
R6K_BASE = '/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K'
r6k_train_hazy = find_dir(R6K_BASE, ['train/hazy', 'RESIDE-6K/train/hazy'])
r6k_train_gt   = find_dir(R6K_BASE, ['train/GT', 'train/gt', 'RESIDE-6K/train/GT'])
r6k_test_hazy  = find_dir(R6K_BASE, ['test/hazy', 'RESIDE-6K/test/hazy'])
r6k_test_gt    = find_dir(R6K_BASE, ['test/GT', 'test/gt', 'RESIDE-6K/test/GT'])

# ─── ITS ───
ITS_BASE = '/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard'
its_hazy = find_dir(ITS_BASE, ['hazy', 'Hazy', 'ITS/hazy'])
its_gt   = find_dir(ITS_BASE, ['clear', 'Clear', 'GT', 'gt', 'ITS/clear'])

# ─── SOTS ───
SOTS_BASE = '/kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside'
sots_in_hazy  = find_dir(SOTS_BASE, ['indoor/hazy', 'SOTS/indoor/hazy'])
sots_in_gt    = find_dir(SOTS_BASE, ['indoor/clear', 'indoor/gt', 'indoor/GT', 'SOTS/indoor/GT'])
sots_out_hazy = find_dir(SOTS_BASE, ['outdoor/hazy', 'SOTS/outdoor/hazy'])
sots_out_gt   = find_dir(SOTS_BASE, ['outdoor/clear', 'outdoor/Clear', 'outdoor/GT', 'SOTS/outdoor/GT'])

# Print verification
print("\n" + "=" * 65)
print("DATASET PATH VERIFICATION")
print("=" * 65)
for name, path in [
    ('R6K Train Hazy', r6k_train_hazy), ('R6K Train GT', r6k_train_gt),
    ('R6K Test Hazy', r6k_test_hazy), ('R6K Test GT', r6k_test_gt),
    ('ITS Hazy', its_hazy), ('ITS GT', its_gt),
    ('SOTS In Hazy', sots_in_hazy), ('SOTS In GT', sots_in_gt),
    ('SOTS Out Hazy', sots_out_hazy), ('SOTS Out GT', sots_out_gt),
]:
    if path:
        n = len(os.listdir(path))
        print(f"  ✓ {name:<18} {path}  [{n} files]")
    else:
        print(f"  ✗ {name:<18} NOT FOUND")
print("=" * 65)




DATASET PATH VERIFICATION
  ✓ R6K Train Hazy     /kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/train/hazy  [6000 files]
  ✓ R6K Train GT       /kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/train/GT  [6000 files]
  ✓ R6K Test Hazy      /kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/test/hazy  [1000 files]
  ✓ R6K Test GT        /kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/test/GT  [1000 files]
  ✓ ITS Hazy           /kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard/hazy  [13990 files]
  ✓ ITS GT             /kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard/clear  [1399 files]
  ✓ SOTS In Hazy       /kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside/indoor/hazy  [500 files]
  ✓ SOTS In GT         /kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside/indoor/clear  [50 files]
  ✓ SOTS Out Hazy      /kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside/outdoor/hazy

In [24]:
# ═══════════════════════════════════════════════════════════════
# SECTION 5 — DATA AUGMENTATION & TF.DATA PIPELINE
# ═══════════════════════════════════════════════════════════════
from pathlib import Path

def get_gt_path(hazy_name, gt_dir):
    """Find corresponding ground truth file for a hazy image."""
    stem = Path(hazy_name).stem
    candidates = [stem, stem.split('_')[0], stem.replace('_hazy', ''),
                  stem.replace('_fog', ''), stem.replace('_synthetic', '')]
    for cand in candidates:
        for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.bmp']:
            p = os.path.join(gt_dir, cand + ext)
            if os.path.exists(p):
                return p
    return None

def preprocess_hazy(img):
    """Exact same preprocessing as Stage 1-3."""
    img = np.power(np.clip(img, 0, 1), 0.9).astype(np.float32)
    for c in range(3):
        lo, hi = img[:, :, c].min(), img[:, :, c].max()
        if hi > lo + 1e-6:
            img[:, :, c] = (img[:, :, c] - lo) / (hi - lo)
    return np.clip(img, 0, 1).astype(np.float32)

def load_and_preprocess(hazy_path, gt_path, size=IMG_SIZE):
    """Load and preprocess a single image pair."""
    hazy = cv2.imread(hazy_path)
    gt = cv2.imread(gt_path)
    if hazy is None or gt is None:
        return None, None
    hazy = cv2.cvtColor(hazy, cv2.COLOR_BGR2RGB)
    gt = cv2.cvtColor(gt, cv2.COLOR_BGR2RGB)
    hazy = cv2.resize(hazy, (size, size)).astype(np.float32) / 255.0
    gt = cv2.resize(gt, (size, size)).astype(np.float32) / 255.0
    hazy = preprocess_hazy(hazy)
    return hazy, gt

def collect_pairs(hazy_dir, gt_dir, limit=None):
    """Collect all valid hazy-GT pairs from a directory."""
    if not hazy_dir or not gt_dir:
        return [], []
    files = sorted([f for f in os.listdir(hazy_dir)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
    if limit:
        files = files[:limit]
    hazy_list, gt_list = [], []
    for fname in files:
        gp = get_gt_path(fname, gt_dir)
        if gp:
            hazy_list.append(os.path.join(hazy_dir, fname))
            gt_list.append(gp)
    return hazy_list, gt_list

# Collect all training pairs
print("\nCollecting training pairs...")
r6k_h, r6k_g = collect_pairs(r6k_train_hazy, r6k_train_gt)
its_h, its_g = collect_pairs(its_hazy, its_gt)

all_hazy = r6k_h + its_h
all_gt   = r6k_g + its_g
print(f"  RESIDE-6K train: {len(r6k_h)} pairs")
print(f"  ITS train      : {len(its_h)} pairs")
print(f"  TOTAL          : {len(all_hazy)} pairs")

# Shuffle
combined = list(zip(all_hazy, all_gt))
random.shuffle(combined)
all_hazy, all_gt = zip(*combined) if combined else ([], [])
all_hazy, all_gt = list(all_hazy), list(all_gt)

# Train/Val split (90/10)
split = int(0.9 * len(all_hazy))
train_hazy, val_hazy = all_hazy[:split], all_hazy[split:]
train_gt, val_gt     = all_gt[:split], all_gt[split:]
print(f"\n  Train: {len(train_hazy)} pairs")
print(f"  Val  : {len(val_hazy)} pairs")




  RESIDE-6K train: 6000 pairs
  ITS train      : 13990 pairs
  TOTAL          : 19990 pairs

  Train: 17991 pairs
  Val  : 1999 pairs


In [25]:
# ═══════════════════════════════════════════════════════════════
# SECTION 6 — TF.DATA GENERATOR WITH AUGMENTATION
# ═══════════════════════════════════════════════════════════════

def data_generator(hazy_paths, gt_paths, augment=False):
    """Generator yielding (hazy, gt) pairs with optional augmentation."""
    indices = list(range(len(hazy_paths)))
    while True:
        random.shuffle(indices)
        for idx in indices:
            hazy, gt = load_and_preprocess(hazy_paths[idx], gt_paths[idx])
            if hazy is None:
                continue

            if augment:
                # Random horizontal flip
                if random.random() > 0.5:
                    hazy = hazy[:, ::-1, :]
                    gt = gt[:, ::-1, :]
                # Random vertical flip
                if random.random() > 0.5:
                    hazy = hazy[::-1, :, :]
                    gt = gt[::-1, :, :]
                # Random 90° rotation
                k = random.randint(0, 3)
                if k > 0:
                    hazy = np.rot90(hazy, k)
                    gt = np.rot90(gt, k)
                # Random brightness jitter (subtle for Stage 4)
                if random.random() > 0.7:
                    factor = random.uniform(0.95, 1.05)
                    hazy = np.clip(hazy * factor, 0, 1).astype(np.float32)
                # MixUp-style blend (very light, 5% chance)
                if random.random() > 0.95:
                    idx2 = random.choice(indices)
                    h2, g2 = load_and_preprocess(hazy_paths[idx2], gt_paths[idx2])
                    if h2 is not None:
                        alpha = random.uniform(0.1, 0.3)
                        hazy = (hazy * (1 - alpha) + h2 * alpha).astype(np.float32)
                        gt = (gt * (1 - alpha) + g2 * alpha).astype(np.float32)

            yield hazy.astype(np.float32), gt.astype(np.float32)

def create_dataset(hazy_paths, gt_paths, batch_size, augment=False):
    """Create tf.data.Dataset from generator."""
    ds = tf.data.Dataset.from_generator(
        lambda: data_generator(hazy_paths, gt_paths, augment=augment),
        output_signature=(
            tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, CHANNELS), dtype=tf.float32),
            tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, CHANNELS), dtype=tf.float32),
        )
    )
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_hazy, train_gt, BATCH_SIZE, augment=True)
val_ds   = create_dataset(val_hazy, val_gt, BATCH_SIZE, augment=False)

train_steps = max(1, len(train_hazy) // BATCH_SIZE)
val_steps   = max(1, len(val_hazy) // BATCH_SIZE)

print(f"\n✓ Datasets created")
print(f"  Train steps/epoch: {train_steps}")
print(f"  Val steps/epoch  : {val_steps}")




✓ Datasets created
  Train steps/epoch: 1124
  Val steps/epoch  : 124


In [26]:
# ═══════════════════════════════════════════════════════════════
# SECTION 7 — STAGE 4 LOSS FUNCTIONS (Perceptual Refinement)
# ═══════════════════════════════════════════════════════════════
# References:
#   - Charbonnier: "A Highly Accurate and Remarkably Efficient Image Dehazing" (Ren et al.)
#   - MS-SSIM: "Multiscale structural similarity for image quality assessment" (Wang et al.)
#   - VGG Perceptual: "Perceptual Losses for Real-Time Style Transfer" (Johnson et al., 2016)
#   - FFT Loss: "Focal Frequency Loss" (Jiang et al., ICCV 2021)
#   - Color Consistency: "AOD-Net" (Li et al., 2017)
#   - Gradient Loss: "Image Super-Resolution via Dual-State Recurrent Networks" (Han et al.)

# ─── VGG19 Feature Extractor ───
def build_vgg19_extractor():
    """Multi-layer VGG19 feature extractor for perceptual loss.
    Uses features from blocks 1-5 for multi-scale perceptual matching.
    """
    vgg = VGG19(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    vgg.trainable = False
    # Extract from multiple layers for richer perceptual signal
    feature_layers = [
        'block1_conv2',  # Low-level edges/textures
        'block2_conv2',  # Medium textures
        'block3_conv4',  # Higher-level patterns
        'block4_conv4',  # Semantic features
        'block5_conv4',  # Deep semantic features
    ]
    outputs = [vgg.get_layer(name).output for name in feature_layers]
    return Model(inputs=vgg.input, outputs=outputs, name='vgg19_extractor')

vgg_extractor = build_vgg19_extractor()
# Layer-wise weights (emphasize mid-level features for dehazing)
VGG_LAYER_WEIGHTS = [0.1, 0.15, 0.3, 0.3, 0.15]
print(f"✓ VGG19 extractor built ({len(vgg_extractor.outputs)} feature layers)")

# ─── Individual Loss Functions ───
def charbonnier_loss(y_true, y_pred, eps=1e-6):
    """Charbonnier loss: smooth L1 alternative, robust to outliers."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.sqrt(tf.square(y_true - y_pred) + eps * eps))

def ssim_loss(y_true, y_pred):
    """1 - SSIM loss for structural similarity optimization."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

def ms_ssim_loss(y_true, y_pred):
    """XLA-safe MS-SSIM approximation using avg pooling to avoid MirrorPad_grad errors."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    loss = 0.0
    # Official MS-SSIM scale weights
    weights = [0.0448, 0.2856, 0.3001, 0.2363, 0.1333] 
    
    # Manually compute SSIM at 5 scales using average pooling (100% XLA Safe)
    for w in weights:
        loss += w * (1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0)))
        y_true = tf.nn.avg_pool2d(y_true, ksize=2, strides=2, padding='VALID')
        y_pred = tf.nn.avg_pool2d(y_pred, ksize=2, strides=2, padding='VALID')
        
    return loss

def vgg_perceptual_loss(y_true, y_pred):
    """Multi-layer VGG19 perceptual loss with layer-wise weighting."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    # VGG expects [0, 255] with ImageNet preprocessing
    y_true_vgg = tf.keras.applications.vgg19.preprocess_input(y_true * 255.0)
    y_pred_vgg = tf.keras.applications.vgg19.preprocess_input(y_pred * 255.0)
    feat_true = vgg_extractor(y_true_vgg, training=False)
    feat_pred = vgg_extractor(y_pred_vgg, training=False)
    loss = 0.0
    for ft, fp, w in zip(feat_true, feat_pred, VGG_LAYER_WEIGHTS):
        loss += w * tf.reduce_mean(tf.abs(ft - fp))
    return loss

def edge_loss(y_true, y_pred):
    """Sobel edge loss: preserves sharp edges and fine structures."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    # Convert to grayscale
    gray_true = tf.image.rgb_to_grayscale(y_true)
    gray_pred = tf.image.rgb_to_grayscale(y_pred)
    # Sobel edges
    sobel_true_x = tf.image.sobel_edges(gray_true)[:, :, :, :, 0]
    sobel_true_y = tf.image.sobel_edges(gray_true)[:, :, :, :, 1]
    sobel_pred_x = tf.image.sobel_edges(gray_pred)[:, :, :, :, 0]
    sobel_pred_y = tf.image.sobel_edges(gray_pred)[:, :, :, :, 1]
    edge_true = tf.sqrt(tf.square(sobel_true_x) + tf.square(sobel_true_y) + 1e-8)
    edge_pred = tf.sqrt(tf.square(sobel_pred_x) + tf.square(sobel_pred_y) + 1e-8)
    return tf.reduce_mean(tf.abs(edge_true - edge_pred))

def fft_loss(y_true, y_pred):
    """Focal Frequency Loss: penalizes frequency-domain differences.
    Based on: 'Focal Frequency Loss for Image Reconstruction and Synthesis' (ICCV 2021)
    This helps recover high-frequency details lost in dehazing.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    # Compute 2D FFT per channel
    fft_true = tf.signal.fft2d(tf.cast(tf.transpose(y_true, [0, 3, 1, 2]), tf.complex64))
    fft_pred = tf.signal.fft2d(tf.cast(tf.transpose(y_pred, [0, 3, 1, 2]), tf.complex64))
    # Magnitude spectrum
    mag_true = tf.abs(fft_true)
    mag_pred = tf.abs(fft_pred)
    # Focal weight: higher weight on hard frequencies
    freq_distance = tf.abs(mag_true - mag_pred)
    weight = freq_distance / (tf.reduce_max(freq_distance) + 1e-8)
    return tf.reduce_mean(weight * freq_distance)

def color_consistency_loss(y_true, y_pred):
    """Color consistency loss: penalizes mean color shift per channel.
    Ensures the dehazed image maintains correct color balance.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    mean_true = tf.reduce_mean(y_true, axis=[1, 2])  # [B, 3]
    mean_pred = tf.reduce_mean(y_pred, axis=[1, 2])  # [B, 3]
    return tf.reduce_mean(tf.square(mean_true - mean_pred))

def gradient_loss(y_true, y_pred):
    """Gradient loss: penalizes differences in image gradients.
    Captures texture and structural consistency.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    # Horizontal and vertical gradients
    dx_true = y_true[:, :, 1:, :] - y_true[:, :, :-1, :]
    dy_true = y_true[:, 1:, :, :] - y_true[:, :-1, :, :]
    dx_pred = y_pred[:, :, 1:, :] - y_pred[:, :, :-1, :]
    dy_pred = y_pred[:, 1:, :, :] - y_pred[:, :-1, :, :]
    return tf.reduce_mean(tf.abs(dx_true - dx_pred)) + tf.reduce_mean(tf.abs(dy_true - dy_pred))


def mse_loss(y_true, y_pred):
    """Mean Squared Error for direct PSNR optimization."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.square(y_true - y_pred))

print("✓ All loss functions defined (including MSE)")



✓ VGG19 extractor built (5 feature layers)
✓ All loss functions defined (including MSE)


In [27]:
# ═══════════════════════════════════════════════════════════════
# SECTION 8 — COMBINED STAGE 4 LOSS & METRICS
# ═══════════════════════════════════════════════════════════════
# Loss weights optimized for perceptual refinement at Stage 4:
#   - Heavy emphasis on structural (SSIM, MS-SSIM) and perceptual (VGG)
#   - Charbonnier for pixel stability
#   - Edge + FFT for high-frequency detail recovery
#   - Color + Gradient for consistency

LOSS_WEIGHTS = {
    'mse':               0.40,
    'charbonnier':       0.40,
    'ssim':              0.05,
    'ms_ssim':           0.10,
    'vgg_perceptual':    0.05,
    'edge':              0.00,
    'fft':               0.00,
    'color_consistency': 0.00,
    'gradient':          0.00,
}

print("Stage 4 Loss Weights:")
for k, v in LOSS_WEIGHTS.items():
    print(f"  {k:<20}: {v:.2f}")
print(f"  {'TOTAL':<20}: {sum(LOSS_WEIGHTS.values()):.2f}")

def stage4_combined_loss(y_true, y_pred):
    """Stage 4 combined perceptual refinement loss."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    loss  = LOSS_WEIGHTS['mse']               * mse_loss(y_true, y_pred)
    loss += LOSS_WEIGHTS['charbonnier']       * charbonnier_loss(y_true, y_pred)
    loss += LOSS_WEIGHTS['ssim']              * ssim_loss(y_true, y_pred)
    loss += LOSS_WEIGHTS['ms_ssim']           * ms_ssim_loss(y_true, y_pred)
    loss += LOSS_WEIGHTS['vgg_perceptual']    * vgg_perceptual_loss(y_true, y_pred)
    loss += LOSS_WEIGHTS['edge']              * edge_loss(y_true, y_pred)
    loss += LOSS_WEIGHTS['fft']               * fft_loss(y_true, y_pred)
    loss += LOSS_WEIGHTS['color_consistency'] * color_consistency_loss(y_true, y_pred)
    loss += LOSS_WEIGHTS['gradient']          * gradient_loss(y_true, y_pred)
    return loss

# ─── Metrics ───
def psnr_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0))

def ssim_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

def mae_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.abs(y_true - y_pred))

print("\n✓ Combined loss function and metrics defined")



Stage 4 Loss Weights:
  mse                 : 0.40
  charbonnier         : 0.40
  ssim                : 0.05
  ms_ssim             : 0.10
  vgg_perceptual      : 0.05
  edge                : 0.00
  fft                 : 0.00
  color_consistency   : 0.00
  gradient            : 0.00
  TOTAL               : 1.00

✓ Combined loss function and metrics defined


In [28]:
# ═══════════════════════════════════════════════════════════════
# SECTION 9 — EXPONENTIAL MOVING AVERAGE (EMA)
# ═══════════════════════════════════════════════════════════════

class EMACallback(tf.keras.callbacks.Callback):
    """Exponential Moving Average of model weights.
    Maintains a shadow copy of weights (EMA-smoothed) and swaps them
    in during validation for more stable metrics. Swaps back after.
    """
    def __init__(self, decay=0.999):
        super().__init__()
        self.decay = decay
        self.ema_weights = None
        self.backup_weights = None

    def on_train_begin(self, logs=None):
        self.ema_weights = [tf.identity(w) for w in self.model.trainable_weights]

    def on_train_batch_end(self, batch, logs=None):
        for i, w in enumerate(self.model.trainable_weights):
            self.ema_weights[i] = self.decay * self.ema_weights[i] + (1.0 - self.decay) * w

    def on_epoch_end(self, epoch, logs=None):
        # Swap in EMA weights for validation
        self.backup_weights = [tf.identity(w) for w in self.model.trainable_weights]
        for i, w in enumerate(self.model.trainable_weights):
            w.assign(self.ema_weights[i])

    def on_test_end(self, logs=None):
        # Swap back original weights after validation
        if self.backup_weights is not None:
            for i, w in enumerate(self.model.trainable_weights):
                w.assign(self.backup_weights[i])
            self.backup_weights = None

    def apply_ema_weights(self):
        """Permanently apply EMA weights (call after training)."""
        if self.ema_weights is not None:
            for i, w in enumerate(self.model.trainable_weights):
                w.assign(self.ema_weights[i])
            print("✓ EMA weights applied permanently")

print("✓ EMA Callback defined (decay={:.4f})".format(EMA_DECAY))



✓ EMA Callback defined (decay=0.9990)


In [29]:
# ═══════════════════════════════════════════════════════════════
# SECTION 10 — OPTIMIZER, LR SCHEDULE & COMPILE
# ═══════════════════════════════════════════════════════════════

# Cosine Decay Restarts Schedule
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=2e-5,
    first_decay_steps=train_steps * 10,  # restart after 10 epochs
    t_mul=2.0,
    m_mul=0.5,
    alpha=1e-7 / 2e-5,
    name='cosine_decay_restarts_s4'
)

# Nadam Optimizer
optimizer = tf.keras.optimizers.Nadam(
    learning_rate=lr_schedule,
    weight_decay=WEIGHT_DECAY,
    clipnorm=CLIP_NORM,
    epsilon=1e-7,
)

# Compile
model.compile(
    optimizer=optimizer,
    loss=stage4_combined_loss,
    metrics=[psnr_metric, ssim_metric, mae_metric],
)

print("✓ Model compiled")
print(f"  Optimizer     : Nadam (clipnorm={CLIP_NORM})")
print(f"  LR Schedule   : CosineDecayRestarts 2e-5")
print(f"  Weight Decay  : {WEIGHT_DECAY}")

✓ Model compiled
  Optimizer     : Nadam (clipnorm=1.0)
  LR Schedule   : CosineDecayRestarts 2e-5
  Weight Decay  : 1e-05


In [30]:
# ═══════════════════════════════════════════════════════════════
# SECTION 11 — CALLBACKS
# ═══════════════════════════════════════════════════════════════
os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/logs', exist_ok=True)

SAVE_PATH = '/kaggle/working/models/stage4_best.keras'

# EMA
ema_cb = EMACallback(decay=EMA_DECAY)

# Model Checkpoint
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=SAVE_PATH,
    monitor='val_psnr_metric',
    mode='max',
    save_best_only=True,
    verbose=1,
)

# Early Stopping
early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_psnr_metric',
    mode='max',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

# CSV Logger
csv_logger_cb = tf.keras.callbacks.CSVLogger(
    '/kaggle/working/logs/stage4_history.csv',
    separator=',',
    append=False,
)

# LR Logger (custom)
class LRLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr_obj = self.model.optimizer.learning_rate

        try:
            lr = float(lr_obj(self.model.optimizer.iterations))
        except TypeError:
            lr = float(tf.keras.backend.get_value(lr_obj))

        if logs is not None:
            logs['lr'] = lr

        print(f"  LR: {lr:.2e}")

lr_logger_cb = LRLogger()

# Progress
class ProgressCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        print(f"\n{'═' * 65}")
        print(f"EPOCH {epoch + 1}/{MAX_EPOCHS} — Stage 4 Transfer Learning")
        print(f"{'═' * 65}")

    def on_epoch_end(self, epoch, logs=None):
        if logs:
            print(f"  Train — Loss: {logs.get('loss', 0):.6f} | "
                  f"PSNR: {logs.get('psnr_metric', 0):.4f} | "
                  f"SSIM: {logs.get('ssim_metric', 0):.4f} | "
                  f"MAE: {logs.get('mae_metric', 0):.6f}")
            print(f"  Val   — Loss: {logs.get('val_loss', 0):.6f} | "
                  f"PSNR: {logs.get('val_psnr_metric', 0):.4f} | "
                  f"SSIM: {logs.get('val_ssim_metric', 0):.4f} | "
                  f"MAE: {logs.get('val_mae_metric', 0):.6f}")

progress_cb = ProgressCallback()

callbacks = [
    ema_cb,
    checkpoint_cb,
    early_stop_cb,
    csv_logger_cb,
    lr_logger_cb,
    progress_cb,
]


print("✓ All callbacks configured")
print(f"  Checkpoint  : {SAVE_PATH} (monitor=val_psnr_metric, mode=max)")
print(f"  Early Stop  : patience={PATIENCE}")
print(f"  CSV Logger  : stage4_history.csv")
print(f"  EMA         : decay={EMA_DECAY}")



✓ All callbacks configured
  Checkpoint  : /kaggle/working/models/stage4_best.keras (monitor=val_psnr_metric, mode=max)
  Early Stop  : patience=8
  CSV Logger  : stage4_history.csv
  EMA         : decay=0.999


In [31]:
print("Train samples:", len(train_hazy))
print("Val samples:", len(val_hazy))

print("Train steps:", train_steps)
print("Val steps:", val_steps)

Train samples: 17991
Val samples: 1999
Train steps: 1124
Val steps: 124


In [32]:
batch = next(iter(train_ds))

print(type(batch))
print(len(batch))
print(batch[0].shape)
print(batch[1].shape)

<class 'tuple'>
2
(16, 256, 256, 3)
(16, 256, 256, 3)


In [33]:
x_batch, y_batch = next(iter(train_ds))

pred = model(x_batch[:1])

print(pred.shape)

(1, 256, 256, 3)


In [34]:
# ═══════════════════════════════════════════════════════════════
# SECTION 12 — STAGE 4 TRAINING
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STARTING STAGE 4 TRAINING — Transfer Learning from Stage 3")
print("=" * 70)
print(f"  Model        : ECA-LDNet ({model.count_params():,} params)")
print(f"  Frozen layers: NONE (all trainable)")
print(f"  Epochs       : {MAX_EPOCHS}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Train steps  : {train_steps}")
print(f"  Val steps    : {val_steps}")
print(f"  LR           : {INIT_LR} → {MIN_LR} (Cosine Decay)")
print(f"  Loss         : 8-component perceptual refinement")
print()

t_start = time.time()

history = model.fit(
    train_ds,
    epochs=MAX_EPOCHS,
    steps_per_epoch=train_steps,
    validation_data=val_ds,
    validation_steps=val_steps,
    callbacks=callbacks,
    initial_epoch=7,
    verbose=1,
)

train_time = time.time() - t_start
print(f"\n{'=' * 70}")
print(f"TRAINING COMPLETE — {train_time / 60:.1f} minutes")
print(f"{'=' * 70}")

# Apply EMA weights permanently
ema_cb.apply_ema_weights()

# Save final model with EMA weights
FINAL_SAVE_PATH = '/kaggle/working/models/stage4_final_ema.keras'
model.save(FINAL_SAVE_PATH)
print(f"✓ Final model (EMA) saved: {FINAL_SAVE_PATH}")




STARTING STAGE 4 TRAINING — Transfer Learning from Stage 3
  Model        : ECA-LDNet (1,481,871 params)
  Frozen layers: NONE (all trainable)
  Epochs       : 30
  Batch size   : 16
  Train steps  : 1124
  Val steps    : 124
  LR           : 1e-05 → 1e-07 (Cosine Decay)
  Loss         : 8-component perceptual refinement


═════════════════════════════════════════════════════════════════
EPOCH 8/30 — Stage 4 Transfer Learning
═════════════════════════════════════════════════════════════════
Epoch 8/30


I0000 00:00:1780923503.499883     125 service.cc:152] XLA service 0x780598057da0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780923503.499940     125 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780923586.966742     125 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1124/1124 ━━━━━━━━━━━━━━━━━━━━ 0s 634ms/step - loss: 1.2013 - mae_metric: 0.0206 - psnr_metric: 32.0420 - ssim_metric: 0.9637
Epoch 8: val_psnr_metric improved from None to 33.87440, saving model to /kaggle/working/models/stage4_best.keras

Epoch 8: finished saving model to /kaggle/working/models/stage4_best.keras
  LR: 1.95e-05
  Train — Loss: 1.196198 | PSNR: 32.0480 | SSIM: 0.9638 | MAE: 0.020658
  Val   — Loss: 1.060624 | PSNR: 33.8744 | SSIM: 0.9697 | MAE: 0.015997
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 948s 707ms/step - loss: 1.1962 - mae_metric: 0.0207 - psnr_metric: 32.0480 - ssim_metric: 0.9638 - val_loss: 1.0606 - val_mae_metric: 0.0160 - val_psnr_metric: 33.8744 - val_ssim_metric: 0.9697 - lr: 1.9513e-05

═════════════════════════════════════════════════════════════════
EPOCH 9/30 — Stage 4 Transfer Learning
═════════════════════════════════════════════════════════════════
Epoch 9/30
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 0s 615ms/step - loss: 1.1973 - mae_metric: 0.0207 - psnr_metric: 32.0

In [1]:
# restart

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 13 — TRAINING HISTORY VISUALIZATION
# ═══════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('/kaggle/working/plots', exist_ok=True)

h = history.history

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Loss
axes[0, 0].plot(h.get('loss', []), 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(h.get('val_loss', []), 'r--', label='Val Loss', linewidth=2)
axes[0, 0].set_title('Stage 4 — Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(alpha=0.3)
axes[0, 0].set_xlabel('Epoch')

# PSNR
axes[0, 1].plot(h.get('psnr_metric', []), 'b-', label='Train PSNR', linewidth=2)
axes[0, 1].plot(h.get('val_psnr_metric', []), 'r--', label='Val PSNR', linewidth=2)
axes[0, 1].set_title('Stage 4 — PSNR (dB)', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(alpha=0.3)
axes[0, 1].set_xlabel('Epoch')

# SSIM
axes[1, 0].plot(h.get('ssim_metric', []), 'b-', label='Train SSIM', linewidth=2)
axes[1, 0].plot(h.get('val_ssim_metric', []), 'r--', label='Val SSIM', linewidth=2)
axes[1, 0].set_title('Stage 4 — SSIM', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(alpha=0.3)
axes[1, 0].set_xlabel('Epoch')

# MAE
axes[1, 1].plot(h.get('mae_metric', []), 'b-', label='Train MAE', linewidth=2)
axes[1, 1].plot(h.get('val_mae_metric', []), 'r--', label='Val MAE', linewidth=2)
axes[1, 1].set_title('Stage 4 — MAE', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(alpha=0.3)
axes[1, 1].set_xlabel('Epoch')

plt.suptitle('ECA-LDNet Stage 4 — Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/plots/stage4_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Training curves saved")



In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 14 — LOAD BEST MODEL FOR EVALUATION
# ═══════════════════════════════════════════════════════════════
# Decide which model to evaluate: checkpoint (best val_psnr) or final EMA
# We'll use the checkpoint first, then compare.

best_model_path = SAVE_PATH  # stage4_best.keras
if os.path.exists(best_model_path):
    print(f"Loading best checkpoint: {best_model_path}")
    eval_model = tf.keras.models.load_model(
        best_model_path, custom_objects=CUSTOM_OBJECTS, compile=False)
    print(f"✓ Best model loaded ({eval_model.count_params():,} params)")
else:
    print("Using current model (EMA weights applied)")
    eval_model = model



In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 15 — EVALUATION FUNCTIONS
# ═══════════════════════════════════════════════════════════════

def predict_full(model, img, patch=IMG_SIZE, tta=True):
    """Full prediction with optional Test-Time Augmentation (4-way)."""
    def _pred(im):
        return model.predict(im[np.newaxis], verbose=0)[0].astype(np.float32)

    out = _pred(img)
    if tta:
        o_lr = _pred(img[:, ::-1, :])[:, ::-1, :]
        o_ud = _pred(img[::-1, :, :])[::-1, :, :]
        o_r = np.rot90(_pred(np.rot90(img, 1)), -1)
        out = (out + o_lr + o_ud + o_r) / 4.0
    return np.clip(out, 0, 1).astype(np.float32)

def compute_psnr(pred, gt):
    return float(tf.image.psnr(pred[np.newaxis], gt[np.newaxis], max_val=1.0))

def compute_ssim(pred, gt):
    return float(tf.image.ssim(pred[np.newaxis], gt[np.newaxis], max_val=1.0))

def compute_mae(pred, gt):
    return float(np.mean(np.abs(pred - gt)))

def compute_ms_ssim(pred, gt):
    try:
        val = float(tf.image.ssim_multiscale(
            pred[np.newaxis], gt[np.newaxis], max_val=1.0))
        return val if not np.isnan(val) else compute_ssim(pred, gt)
    except:
        return compute_ssim(pred, gt)

def evaluate_dataset(model, hazy_dir, gt_dir, name, limit=None, tta=True, save_csv=None):
    """Evaluate model on a full dataset with all metrics."""
    if not hazy_dir or not gt_dir:
        print(f"  SKIP {name} — dataset not found")
        return None
    files = sorted([f for f in os.listdir(hazy_dir)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
    if limit:
        files = files[:limit]
    print(f"\n{'─' * 60}")
    print(f"Evaluating [{name}] — {len(files)} images, TTA={tta}")
    print(f"{'─' * 60}")

    records = []
    t_start = time.time()
    for i, fname in enumerate(files):
        hp = os.path.join(hazy_dir, fname)
        gp = get_gt_path(fname, gt_dir)
        if not gp:
            continue
        hazy, gt = load_and_preprocess(hp, gp)
        if hazy is None:
            continue

        pred = predict_full(model, hazy, tta=tta)

        psnr = compute_psnr(pred, gt)
        ssim = compute_ssim(pred, gt)
        mae = compute_mae(pred, gt)
        ms_ssim = compute_ms_ssim(pred, gt)

        records.append({
            'file': fname, 'PSNR': psnr, 'SSIM': ssim,
            'MAE': mae, 'MS_SSIM': ms_ssim,
        })

        if (i + 1) % 50 == 0 or (i + 1) == len(files):
            avg_p = np.mean([r['PSNR'] for r in records])
            avg_s = np.mean([r['SSIM'] for r in records])
            elapsed = time.time() - t_start
            print(f"  [{i + 1:>4}/{len(files)}] PSNR={avg_p:.4f} SSIM={avg_s:.4f} ({elapsed:.1f}s)")

    if not records:
        print(f"  No valid pairs found for {name}")
        return None

    psnrs = [r['PSNR'] for r in records]
    ssims = [r['SSIM'] for r in records]
    maes  = [r['MAE'] for r in records]
    ms_ssims = [r['MS_SSIM'] for r in records]

    print(f"\n  ╔═══ {name} RESULTS ═══")
    print(f"  ║ Images    : {len(records)}")
    print(f"  ║ PSNR      : {np.mean(psnrs):.4f} ± {np.std(psnrs):.4f} dB")
    print(f"  ║ SSIM      : {np.mean(ssims):.4f} ± {np.std(ssims):.4f}")
    print(f"  ║ MAE       : {np.mean(maes):.6f} ± {np.std(maes):.6f}")
    print(f"  ║ MS-SSIM   : {np.mean(ms_ssims):.4f} ± {np.std(ms_ssims):.4f}")
    print(f"  ╚{'═' * 50}")

    if save_csv:
        os.makedirs(os.path.dirname(save_csv), exist_ok=True)
        with open(save_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=records[0].keys())
            writer.writeheader()
            writer.writerows(records)
        print(f"  Saved: {save_csv}")

    return records

print("✓ Evaluation functions defined")



In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 16 — EVALUATE ON ALL BENCHMARKS
# ═══════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("EVALUATING STAGE 4 MODEL ON ALL BENCHMARKS")
print("═" * 70)

os.makedirs('/kaggle/working/test_results/csv', exist_ok=True)

all_results = {}

# RESIDE-6K Test
r6k_results = evaluate_dataset(
    eval_model, r6k_test_hazy, r6k_test_gt,
    'RESIDE-6K', tta=True,
    save_csv='/kaggle/working/test_results/csv/stage4_results_reside6k.csv')
if r6k_results:
    all_results['RESIDE-6K'] = r6k_results

# SOTS Indoor
sots_in_results = evaluate_dataset(
    eval_model, sots_in_hazy, sots_in_gt,
    'SOTS Indoor', tta=True,
    save_csv='/kaggle/working/test_results/csv/stage4_results_sots_indoor.csv')
if sots_in_results:
    all_results['SOTS Indoor'] = sots_in_results

# SOTS Outdoor
sots_out_results = evaluate_dataset(
    eval_model, sots_out_hazy, sots_out_gt,
    'SOTS Outdoor', tta=True,
    save_csv='/kaggle/working/test_results/csv/stage4_results_sots_outdoor.csv')
if sots_out_results:
    all_results['SOTS Outdoor'] = sots_out_results

# Cleanup
gc.collect()



In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 17 — FINAL SUMMARY & STAGE COMPARISON
# ═══════════════════════════════════════════════════════════════

print("\n" + "═" * 80)
print("  STAGE 4 — FINAL RESULTS SUMMARY")
print("═" * 80)

# Stage 3 baseline (from your results)
stage3_results = {
    'SOTS Indoor':  {'PSNR': 30.5775, 'SSIM': 0.9635},
    'SOTS Outdoor': {'PSNR': 28.9962, 'SSIM': 0.9667},
    'RESIDE-6K':    {'PSNR': 29.9263, 'SSIM': 0.9641},
}

for name, records in all_results.items():
    if records:
        avg_psnr = np.mean([r['PSNR'] for r in records])
        avg_ssim = np.mean([r['SSIM'] for r in records])
        avg_ms_ssim = np.mean([r['MS_SSIM'] for r in records])
        avg_mae = np.mean([r['MAE'] for r in records])

        s3 = stage3_results.get(name, {})
        s3_psnr = s3.get('PSNR', 0)
        s3_ssim = s3.get('SSIM', 0)

        psnr_gain = avg_psnr - s3_psnr if s3_psnr > 0 else 0
        ssim_gain = avg_ssim - s3_ssim if s3_ssim > 0 else 0

        print(f"\n  ┌─── {name} ───")
        print(f"  │ PSNR     : {avg_psnr:.4f} dB  (Stage3: {s3_psnr:.4f}, Δ={psnr_gain:+.4f})")
        print(f"  │ SSIM     : {avg_ssim:.4f}      (Stage3: {s3_ssim:.4f}, Δ={ssim_gain:+.4f})")
        print(f"  │ MS-SSIM  : {avg_ms_ssim:.4f}")
        print(f"  │ MAE      : {avg_mae:.6f}")
        print(f"  └{'─' * 60}")

print(f"\n  Model Parameters: {eval_model.count_params():,} ({eval_model.count_params()/1e6:.3f}M)")

# ─── Save comparison report ───
report_path = '/kaggle/working/test_results/stage4_report.txt'
with open(report_path, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("ECA-LDNet — Stage 4 Transfer Learning Report\n")
    f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 70 + "\n\n")

    f.write("TRAINING CONFIGURATION\n")
    f.write(f"  Base Model    : stage3_best.keras\n")
    f.write(f"  Parameters    : {eval_model.count_params():,} ({eval_model.count_params()/1e6:.3f}M)\n")
    f.write(f"  Optimizer     : AdamW (LR={INIT_LR}, WD={WEIGHT_DECAY})\n")
    f.write(f"  LR Schedule   : CosineDecay {INIT_LR} → {MIN_LR}\n")
    f.write(f"  Epochs        : {MAX_EPOCHS} (with early stopping, patience={PATIENCE})\n")
    f.write(f"  Batch Size    : {BATCH_SIZE}\n")
    f.write(f"  EMA Decay     : {EMA_DECAY}\n")
    f.write(f"  Grad Clip     : clipnorm={CLIP_NORM}\n")
    f.write(f"  Mixed Prec.   : float16\n\n")

    f.write("LOSS FUNCTION\n")
    for k, v in LOSS_WEIGHTS.items():
        f.write(f"  {k:<20}: {v:.2f}\n")
    f.write("\n")

    f.write("BENCHMARK RESULTS\n")
    f.write("-" * 60 + "\n")
    for name, records in all_results.items():
        if records:
            avg_psnr = np.mean([r['PSNR'] for r in records])
            avg_ssim = np.mean([r['SSIM'] for r in records])
            avg_ms_ssim = np.mean([r['MS_SSIM'] for r in records])
            avg_mae = np.mean([r['MAE'] for r in records])
            f.write(f"  {name}:\n")
            f.write(f"    PSNR    : {avg_psnr:.4f} dB\n")
            f.write(f"    SSIM    : {avg_ssim:.4f}\n")
            f.write(f"    MS-SSIM : {avg_ms_ssim:.4f}\n")
            f.write(f"    MAE     : {avg_mae:.6f}\n\n")

    f.write("=" * 70 + "\n")

print(f"\n✓ Report saved: {report_path}")

# ─── Save combined training history with stage info ───
csv_in = '/kaggle/working/logs/stage4_history.csv'
csv_out = '/kaggle/working/test_results/csv/stage4_history_with_stage.csv'

if os.path.exists(csv_in):
    rows = []
    with open(csv_in, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            row['stage'] = 4
            row['global_epoch'] = int(row['epoch']) + 100  # after S1(60)+S2(30)+S3(40)=130
            rows.append(row)
    if rows:
        with open(csv_out, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
        print(f"✓ History CSV saved: {csv_out}")



In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 18 — VISUAL COMPARISON (Best/Worst Samples)
# ═══════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def plot_visual_comparison(model, hazy_dir, gt_dir, save_path, title, tta=True, n=6):
    """Generate best/worst visual comparison grid."""
    if not hazy_dir or not gt_dir:
        return
    files = sorted(os.listdir(hazy_dir))
    random.shuffle(files)
    samples = []
    for fname in files[:200]:
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            continue
        gp = get_gt_path(fname, gt_dir)
        if not gp:
            continue
        hazy, gt = load_and_preprocess(os.path.join(hazy_dir, fname), gp)
        if hazy is None:
            continue
        pred = predict_full(model, hazy, tta=tta)
        pv = compute_psnr(pred, gt)
        sv = compute_ssim(pred, gt)
        samples.append((pv, sv, hazy, pred, gt, fname))
        if len(samples) >= 30:
            break

    if len(samples) < 6:
        print(f"  Not enough samples for {title}")
        return

    samples.sort(key=lambda x: x[0])
    show = samples[:3] + samples[-3:]
    labels = ['Worst'] * 3 + ['Best'] * 3

    fig, axes = plt.subplots(6, 4, figsize=(20, 30))
    for row, (pv, sv, hazy, pred, gt, fname) in enumerate(show):
        error = np.abs(pred - gt)
        axes[row, 0].imshow(np.clip(hazy, 0, 1))
        axes[row, 0].set_title(f'[{labels[row]}] Hazy\n{fname}', fontsize=9)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(np.clip(pred, 0, 1))
        axes[row, 1].set_title(f'Predicted\nPSNR={pv:.2f} SSIM={sv:.4f}', fontsize=9)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(np.clip(gt, 0, 1))
        axes[row, 2].set_title('Ground Truth', fontsize=9)
        axes[row, 2].axis('off')

        im = axes[row, 3].imshow(error, cmap='hot', vmin=0, vmax=0.3)
        axes[row, 3].set_title(f'Error Map\nMAE={np.mean(error):.4f}', fontsize=9)
        axes[row, 3].axis('off')
        plt.colorbar(im, ax=axes[row, 3], fraction=0.046, pad=0.04)

    plt.suptitle(f'{title}\n3 Worst (top) vs 3 Best (bottom) — Stage 4', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Visual comparison saved: {save_path}")

os.makedirs('/kaggle/working/test_results/plots', exist_ok=True)

# Generate visuals
if r6k_test_hazy and r6k_test_gt:
    plot_visual_comparison(eval_model, r6k_test_hazy, r6k_test_gt,
                           '/kaggle/working/test_results/plots/stage4_visual_reside6k.png',
                           'RESIDE-6K — Stage 4')

if sots_in_hazy and sots_in_gt:
    plot_visual_comparison(eval_model, sots_in_hazy, sots_in_gt,
                           '/kaggle/working/test_results/plots/stage4_visual_sots_indoor.png',
                           'SOTS Indoor — Stage 4')

if sots_out_hazy and sots_out_gt:
    plot_visual_comparison(eval_model, sots_out_hazy, sots_out_gt,
                           '/kaggle/working/test_results/plots/stage4_visual_sots_outdoor.png',
                           'SOTS Outdoor — Stage 4')

print("\n✓ All visual comparisons generated")



In [ ]:
# ═══════════════════════════════════════════════════════════════
# SECTION 19 — COMPLETION
# ═══════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  STAGE 4 TRAINING & EVALUATION COMPLETE ✓")
print("═" * 70)
print(f"  Model       : ECA-LDNet ({eval_model.count_params():,} params, {eval_model.count_params()/1e6:.3f}M)")
print(f"  Train Time  : {train_time / 60:.1f} minutes")
print()

for name, records in all_results.items():
    if records:
        avg_p = np.mean([r['PSNR'] for r in records])
        avg_s = np.mean([r['SSIM'] for r in records])
        avg_ms = np.mean([r['MS_SSIM'] for r in records])
        print(f"  {name:<15}: PSNR={avg_p:.4f} dB  SSIM={avg_s:.4f}  MS-SSIM={avg_ms:.4f}")

print()
print("  Saved files:")
print("    /kaggle/working/models/stage4_best.keras")
print("    /kaggle/working/models/stage4_final_ema.keras")
print("    /kaggle/working/logs/stage4_history.csv")
print("    /kaggle/working/test_results/stage4_report.txt")
print("    /kaggle/working/test_results/csv/stage4_*.csv")
print("    /kaggle/working/test_results/plots/stage4_*.png")
print("═" * 70)
print("  DONE ✓")

